In [ ]:
# Assignment2: data cleaning
# Given the information collected in the previous assignment, address the problem
# related to the missing data (if any) and integrate the additional data (if any).

In [3]:
import json
import csv

In [8]:
#Remove the row where 'lyrics' is empty

input_file = 'track_after_assigment1.csv'
output_file = 'removeLyrics.csv'
columnToCheck = 'lyrics_clean'

def cleanEmptyLyrics(input_file, output_file, columnToCheck):
    removed_rows = 0

    with open(input_file, mode='r', newline='', encoding='utf-8') as infile:
        reader = csv.DictReader(infile)
        campi = reader.fieldnames

        with open(output_file, mode='w', newline='', encoding='utf-8') as outfile:
            writer = csv.DictWriter(outfile, fieldnames=campi)
            writer.writeheader()

            for riga in reader:
                lyricsValue = riga.get(columnToCheck, '')

                if lyricsValue.strip() != "":  # remove rows with empty lyrics
                    writer.writerow(riga)
                else:
                    removed_rows += 1

    print(f"Cleaning completed. Data saved to: {output_file}")
    print(f"Rows removed with empty '{columnToCheck}': {removed_rows}")

# execute the function
cleanEmptyLyrics(input_file, output_file, columnToCheck)


Cleaning completed. Data saved to: removeLyrics.csv
Rows removed with empty 'lyrics_clean': 3


In [14]:
import re 

input_file1 = 'removeLyrics.csv' # File from the previous cleaning step
output_file1 = 'tokenandsentenceClean.csv'
columnsToFill = ['n_sentences', 'n_tokens', 'char_per_tok', 'avg_token_per_clause']

# --- Core Logic Functions ---

def calculate_metrics(lyrics_clean: str) -> dict:
    """
    Calculates the linguistic metrics based on the song lyrics.
    This function uses simple string manipulation as external libraries are forbidden.
    
    Args:
        lyrics: The string containing the full song lyrics.
        
    Returns:
        A dictionary with the calculated values for the missing columns.
    """
    if not lyrics_clean or lyrics_clean.strip() == '':
        # Return zeros if lyrics are missing, though this should be rare for the 76 rows we target, we cleaned before.
        return {
            'n_sentences': 0, 
            'n_tokens': 0, 
            'char_per_tok': 0.0, 
            'avg_token_per_clause': 0.0
        }
    
    # Tokenization (Counting Words/Tokens) --> to fill 'n_tokens'
    # Use a regex to split the text by any sequence of non-alphanumeric characters (including spaces, punctuation, etc.)
    # and filter out empty strings resulting from the split.
    words = [token for token in re.split(r'[^a-zA-Z0-9]+', lyrics_clean) if token]
    n_tokens = len(words)

    # Character Count
    total_chars = sum(len(word) for word in words)

    # Sentence Count
    # Assuming sentences are separated by newline characters (the simplest heuristic)
    # This might overestimate or underestimate the true sentence count.
    sentences = [s for s in lyrics_clean.split('\n') if s.strip() != '']
    n_sentences = len(sentences)
    
    # 4. Calculate Derived Metrics
    
    # Average characters per token
    char_per_tok = total_chars / n_tokens if n_tokens > 0 else 0.0
    
    # Average tokens per clause (using sentences as clauses)
    avg_token_per_clause = n_tokens / n_sentences if n_sentences > 0 else 0.0
    
    return {
        'n_sentences': n_sentences, 
        'n_tokens': n_tokens, 
        'char_per_tok': round(char_per_tok, 4), # Rounding for clean output
        'avg_token_per_clause': round(avg_token_per_clause, 4)
    }

# --- Main Processing ---

def fill_missing_metrics(input_file: str, output_file: str):
    """
    Main function to read the CSV, fill the missing metrics using lyrics, and write the new CSV.
    """
    updated_rows_count = 0
    
    try:
        with open(input_file, mode='r', newline='', encoding='utf-8') as infile:
            reader = csv.DictReader(infile)
            fieldnames = reader.fieldnames # Get the original header
            
            # Read all rows into memory for easier manipulation (assuming the file is not excessively large)
            data = list(reader)
            
    except FileNotFoundError:
        print(f"ERROR: Input file not found at {input_file}")
        return

    # Process Data
    for row in data:
        # Check if the row is one of the 76 missing ones (assuming missing values were left as empty strings)
        # We also check if 'lyrics' is NOT missing, otherwise we cannot calculate
        missing_values = ['', 'NaN', 'nan', None]
        is_metrics_missing = any(row.get(col) in missing_values for col in columnsToFill)
        has_lyrics = row.get('lyrics_clean', '').strip() not in ['', 'NaN', 'nan']
        
        if is_metrics_missing and has_lyrics:
            
            # Calculate the new metrics using the available lyrics
            new_metrics = calculate_metrics(row['lyrics_clean'])
            
            # Update the row with the calculated values
            for col, value in new_metrics.items():
                row[col] = str(value) # CSV writers expect string values
            
            updated_rows_count += 1
            
        elif is_metrics_missing and not has_lyrics:
             # In a production setting, you'd log or handle this case. 
             # Since we only removed 3 'lyrics' rows, this shouldn't affect the other 76 rows.
             pass 

    # Write the updated data to the new file
    with open(output_file, mode='w', newline='', encoding='utf-8') as outfile:
        writer = csv.DictWriter(outfile, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(data)

    print(f" Data imputation completed.")
    print(f" Total rows updated with calculated metrics: {updated_rows_count}")
    print(f" Cleaned and imputed data saved to: {output_file}")

# Execute the main function
fill_missing_metrics(input_file1, output_file1)

 Data imputation completed.
 Total rows updated with calculated metrics: 73
 Cleaned and imputed data saved to: tokenandsentenceClean.csv


In [17]:
!pip install langdetect

In [19]:
from langdetect import detect
from langdetect.lang_detect_exception import LangDetectException

# --- Configuration ---
# Define the names of your columns
LYRICS_COL = 'lyrics_clean'
LANGUAGE_COL = 'language'
INPUT_FILE = 'tokenandsentenceClean.csv'
OUTPUT_FILE = 'languageclean.csv'

def detect_language_safe(text):
    """
    Detects the language of a given text after essential preprocessing.
    """
    if not text or isinstance(text, float):
        return 'undetermined'
    
    # 1. Convert to string and Lowercase
    cleaned_text = str(text).lower()
    
    # 2. Remove symbols, punctuation, and numbers
    # Pattern to keep only letters and spaces (a-z)
    cleaned_text = re.sub(r'[^a-z\s]', '', cleaned_text)
    
    # 3. Clean up extra whitespaces (replaces multiple spaces/tabs with one space)
    cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()
    
    # Check if the text is empty after cleaning
    if not cleaned_text:
        return 'undetermined'
        
    try:
        return detect(cleaned_text)
    except LangDetectException:
        return 'undetermined'
    except Exception as e:
        print(f"An unexpected error occurred. Error: {e}")
        return 'error'

# --- Main processing logic using only standard Python types and the csv module ---
def process_lyrics_language(input_filepath, output_filepath):
    """
    Reads a CSV, detects the language for missing values in the language column,
    and writes the results to a new CSV file.
    """
    # This list will hold standard Python dictionaries (the rows)
    processed_rows = [] 
    
    # 1. Read the data
    with open(input_filepath, mode='r', encoding='utf-8', newline='') as infile:
        # DictReader treats rows as standard Python dictionaries
        reader = csv.DictReader(infile)
        fieldnames = reader.fieldnames # Get original column headers
        
        # Check column existence (standard Python check)
        if LYRICS_COL not in fieldnames or LANGUAGE_COL not in fieldnames:
            print(f"Error: CSV must contain '{LYRICS_COL}' and '{LANGUAGE_COL}' columns.")
            return

        print("Starting language detection...")
        
        # 2. Process the data row by row
        for row in reader:
            # Check if 'language' is empty (standard string check)
            if not row.get(LANGUAGE_COL, '').strip():
                lyrics = row.get(LYRICS_COL, '')
                
                detected_lang = detect_language_safe(lyrics)
                
                # Directly update the Python dictionary
                row[LANGUAGE_COL] = detected_lang
                
            processed_rows.append(row)
            
    # 3. Write the results
    with open(output_filepath, mode='w', encoding='utf-8', newline='') as outfile:
        # DictWriter writes standard Python dictionaries
        writer = csv.DictWriter(outfile, fieldnames=fieldnames)
        
        writer.writeheader()
        writer.writerows(processed_rows)
        
    print(f"\nProcessing complete! Results saved to '{output_filepath}'")

# Execute the processing function
process_lyrics_language(INPUT_FILE, OUTPUT_FILE)

Starting language detection...

Processing complete! Results saved to 'languageclean.csv'


In [26]:
import csv

input_file = 'languageclean.csv'  # the file cleaned for lyrics
output_file = 'AudioNaNremoved.csv'
audio_features = ['bpm', 'rolloff', 'flux', 'rms', 'flatness', 'spectral_complexity', 'pitch', 'loudness']

def cleanMissingAudioFeatures(input_file, output_file, columnsToCheck):
    removed_rows = 0

    with open(input_file, mode='r', newline='', encoding='utf-8') as infile:
        reader = csv.DictReader(infile)
        fieldnames = reader.fieldnames

        with open(output_file, mode='w', newline='', encoding='utf-8') as outfile:
            writer = csv.DictWriter(outfile, fieldnames=fieldnames)
            writer.writeheader()

            for row in reader:
                # Check if any of the audio feature columns are empty or null
                if all(row.get(col, '').strip() != '' for col in columnsToCheck):
                    writer.writerow(row)
                else:
                    removed_rows += 1

    print(f"Cleaning completed. Data saved to: {output_file}")
    print(f"Rows removed with missing audio features: {removed_rows}")

# Execute the function
cleanMissingAudioFeatures(input_file, output_file, audio_features)


Cleaning completed. Data saved to: AudioNaNremoved.csv
Rows removed with missing audio features: 64


In [37]:
import urllib.request
import urllib.parse
import base64
import time

# --- Configuration and Constants ---

INPUT_FILE = 'AudioNaNremoved.csv' 
OUTPUT_FILE = 'SpotifyApi.csv'
MISSING_VALUES = ['', 'NaN', 'nan'] 

# !!! REPLACE THESE WITH YOUR ACTUAL SPOTIFY CREDENTIALS !!!
SPOTIFY_CLIENT_ID = "5838810d86504d6bbc6947b39b82d8f7" 
SPOTIFY_CLIENT_SECRET = "b467db2aef154993a7610f779203bc87" 

# Spotify API Endpoints
SPOTIFY_TOKEN_URL = "https://accounts.spotify.com/api/token"
SPOTIFY_SEARCH_URL = "https://api.spotify.com/v1/search"
SPOTIFY_FEATURES_URL = "https://api.spotify.com/v1/audio-features"

# Columns to be Imputed
ALBUM_COLUMNS = ['album_name', 'album_release_date', 'album_type', 'disc_number', 'track_number', 'duration_ms', 'explicit', 'popularity', 'id_album']

# --- 1. Spotify Authentication ---

def get_spotify_token(client_id: str, client_secret: str) -> str | None:
    """
    Obtains an Access Token from Spotify using the Client Credentials Flow.
    This token is required for all subsequent API requests.
    """
    try:
        # Base64 encode the Client ID and Secret for authentication header
        auth_string = f"{client_id}:{client_secret}"
        encoded_auth = base64.b64encode(auth_string.encode('utf-8')).decode('utf-8')

        headers = {
            "Authorization": f"Basic {encoded_auth}",
            "Content-Type": "application/x-www-form-urlencoded"
        }
        data = urllib.parse.urlencode({'grant_type': 'client_credentials'}).encode('utf-8')

        req = urllib.request.Request(SPOTIFY_TOKEN_URL, data=data, headers=headers, method='POST')
        
        with urllib.request.urlopen(req) as response:
            token_info = json.loads(response.read().decode('utf-8'))
            return token_info.get('access_token')
            
    except urllib.error.HTTPError as e:
        print(f"Error getting Spotify token: HTTP {e.code} - {e.reason}")
        return None
    except Exception as e:
        print(f"Error getting Spotify token: {e}")
        return None

# --- 2. Spotify Data Fetching ---

def fetch_spotify_data(title: str, artist: str, token: str) -> dict:
    """
    Searches Spotify for the track ID and then fetches its metadata and audio features.
    
    Returns:
        A dictionary containing track metadata and audio features, or {} on failure.
    """
    headers = {"Authorization": f"Bearer {token}"}
    
    # --- PHASE A: Search for Track ID ---
    
    try:
        search_query = f"track:{title} artist:{artist}"
        encoded_query = urllib.parse.quote(search_query)
        search_url = f"{SPOTIFY_SEARCH_URL}?q={encoded_query}&type=track&limit=1"
        
        req = urllib.request.Request(search_url, headers=headers)
        with urllib.request.urlopen(req) as response:
            search_results = json.loads(response.read().decode('utf-8'))
        
        items = search_results.get('tracks', {}).get('items', [])
        
        if not items:
            return {} # Track not found
        
        # Extract ID and initial metadata
        track_info = items[0]
        album_info = track_info.get('album', {})
        
        return {
            'duration_ms': track_info.get('duration_ms'),
            'explicit': str(track_info.get('explicit')),
            'popularity': track_info.get('popularity'),
            'album_name': album_info.get('name'),
            'album_release_date': album_info.get('release_date'),
            'album_type': album_info.get('album_type'),
            'disc_number': track_info.get('disc_number'),
            'track_number': track_info.get('track_number'),
            'id_album': album_info.get('id')
        }

    except urllib.error.HTTPError as e:
        # Rate limit or other HTTP error
        print(f"DEBUG: Search failed for '{title}': HTTP {e.code}")
        if e.code == 429: # Too Many Requests
            print("WARNING: Spotify Rate Limit hit. Pausing for 5 seconds...")
            time.sleep(5)
        return {}
    except Exception:
        return {} # Parsing or network error

# --- 3. Main Imputation Logic ---

def impute_external_data_spotify(input_file: str, output_file: str):
    """
    Implements the conservative imputation using the Spotify API.
    Only fills the fields if data is successfully fetched; otherwise, preserves original value.
    """
    print("--- Starting Spotify API Imputation ---")
    
    # 1. Get Spotify Access Token
    spotify_token = get_spotify_token(SPOTIFY_CLIENT_ID, SPOTIFY_CLIENT_SECRET)
    if not spotify_token:
        print("FATAL: Could not retrieve Spotify token. Cannot proceed with API calls.")
        return

    imputed_count = 0
    data_to_write = []

    try:
        with open(input_file, mode='r', newline='', encoding='utf-8') as infile:
            reader = csv.DictReader(infile)
            fieldnames = reader.fieldnames
            data = list(reader) 
            
    except FileNotFoundError:
        print(f"ERROR: Input file not found at {input_file}.")
        return

    # 2. Iterate and Impute
    for i, row in enumerate(data):
        # Check if ANY external column is missing
        is_missing_external = any(row.get(col, '').strip().lower() in MISSING_VALUES for col in ALBUM_COLUMNS)
        
        if is_missing_external and row.get('title') and row.get('primary_artist'):
            
            # Fetch data (will be {} if not found)
            external_data = fetch_spotify_data(row['title'], row['primary_artist'], spotify_token)
            updated = False
            
            # Impute the missing fields ONLY IF external_data has a specific value
            for col in ALBUM_COLUMNS:
                current_value = str(row.get(col, '')).strip().lower()
                
                # Check 1: Is the current value missing? 
                # Check 2: Did Spotify successfully return a non-None value for this column?
                if current_value in MISSING_VALUES and col in external_data and external_data[col] is not None:
                    row[col] = str(external_data[col]) # Update the row
                    updated = True
            
            if updated:
                imputed_count += 1
        
        data_to_write.append(row)
        
        # Add a delay to respect Spotify's rate limits
        if i % 10 == 0 and i > 0:
             time.sleep(0.1) 

    # 3. Write the updated data to the new file
    with open(output_file, mode='w', newline='', encoding='utf-8') as outfile:
        writer = csv.DictWriter(outfile, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(data_to_write)

    print(f"\n Spotify Imputation Complete.")
    print(f" Total rows successfully imputed: {imputed_count}")
    print(f" Data saved to: {output_file}")

# --- Execution ---

impute_external_data_spotify(INPUT_FILE, OUTPUT_FILE)

--- Starting Spotify API Imputation ---

 Spotify Imputation Complete.
 Total rows successfully imputed: 71
 Data saved to: SpotifyApi.csv


In [40]:
#After the spotify api imputation, we have still 7 missing values in the album features, we removed those rows.
input_file = 'SpotifyApi.csv'  # file cleaned for lyrics
output_file = 'SpotifyApiNomissin.csv'

# Columns to check for missing values
album_columns_to_check = [
    'album_name', 'album_release_date', 'album_type', 
    'disc_number', 'track_number', 'duration_ms', 
    'explicit', 'popularity', 'id_album'
]

def cleanMissingAlbumData(input_file, output_file, columnsToCheck):
    removed_rows = 0
    cleaned_features_count = 0

    with open(input_file, mode='r', newline='', encoding='utf-8') as infile:
        reader = csv.DictReader(infile)
        fieldnames = reader.fieldnames

        with open(output_file, mode='w', newline='', encoding='utf-8') as outfile:
            writer = csv.DictWriter(outfile, fieldnames=fieldnames)
            writer.writeheader()

            for row in reader:
                if not row.get('featured_artists', '').strip() or row.get('featured_artists', '').lower() in ['nan']:
                    row['featured_artists'] = "NoFeature"
                    cleaned_features_count += 1

                # Keep row only if none of the checked columns are missing
                if all(row.get(col, '').strip() != '' for col in columnsToCheck):
                    writer.writerow(row)
                else:
                    removed_rows += 1

    print(f"Cleaning completed. Data saved to: {output_file}")
    print(f"Rows removed with missing album/track data: {removed_rows}")
    print(f"Featured artist entries cleaned: {cleaned_features_count}")

# Execute the function
cleanMissingAlbumData(input_file, output_file, album_columns_to_check)


Cleaning completed. Data saved to: SpotifyApiNomissin.csv
Rows removed with missing album/track data: 7
Featured artist entries cleaned: 7599
